In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
import time
import chardet
import os


In [2]:

# 실거래 원본
sales_file = "서울시_상권분석_매출_행정동.csv"

# 행정동ID-행정동코드-법정동코드 매핑
edm_mapping_name = "서울시_행정동ID_행정동코드_맵핑.csv"

# 기준날자|행정동코드|평균거래금액|거래건수
sales_name = "서울시_상권분석_매출_행정동_총합_20211_20254.csv"


In [3]:

# 법정동 갯수는 467개
# 행정동 갯수는 426개

edm_mapping_df = pd.read_csv(edm_mapping_name, encoding="utf-8-sig")


#print(f"맵핑 갯수:{len(edm_mapping_df)}, 행정동ID 갯수:{len(edm_mapping_df['행정동_ID'].unique())}, 행정동코드 갯수:{len(edm_mapping_df['행정동코드'].unique())}, 법정동코드 갯수: {len(edm_mapping_df['법정동코드'].unique())}")

print(f"맵핑 갯수:{len(edm_mapping_df)}, 행정동ID 갯수:{len(edm_mapping_df['행정동_ID'].unique())}, 행정동코드 갯수:{len(edm_mapping_df['행정동코드'].unique())}")


edm_mapping_df.head(2)


맵핑 갯수:426, 행정동ID 갯수:426, 행정동코드 갯수:426


,행정동_ID,행정동코드,행정동이름
0,11010720,1111051500,청운효자동
1,11010530,1111053000,사직동


In [14]:
# 분기별 행정동 매출을 sum하고 월별로  interpolate 한다.
#
# 1. 기준_년분기_코드, 행정동_코드 기준으로 당월_매출_금액, 당월_매출_건수, 주중_매출_금액,
# 주말_매출_금액, 남성_매출_금액, 여성_매출_금액
# 2. 기준_년분기_코드를 YYYYMM으로 분리
# 3. fillna, interpolate
# 4. YYYYMM, 행정동_코드 기준으로 소팅


sales_df = pd.read_csv(sales_file,
                           #encoding="cp949"
                           encoding="utf-8-sig"
                           ).sort_values(["기준_년분기_코드","행정동_코드"])


cols = [
    "기준_년분기_코드",
    "행정동_코드",
    "행정동_코드_명",
    "서비스_업종_코드",
    "서비스_업종_코드_명",
    "당월_매출_금액",
    "당월_매출_건수",
    "주중_매출_금액",
    "주말_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액"
]


sales_df = sales_df[sales_df["기준_년분기_코드"] >= 20242]

sales_df = sales_df[cols]

sales_df["행정동코드"] = sales_df["행정동_코드"] * 100



sales_df.to_csv("서울시_행정동_매출_임시.csv",
                encoding="utf-8-sig",
                index=False)

print(f"갯수: {len(sales_df)}")
sales_df.head(2)


갯수: 117969


,기준_년분기_코드,행정동_코드,행정동_코드_명,서비스_업종_코드,서비스_업종_코드_명,당월_매출_금액,당월_매출_건수,주중_매출_금액,주말_매출_금액,남성_매출_금액,여성_매출_금액,행정동코드
376624,20242,11110515,청운효자동,CS100003,일식음식점,420482695.0,17245.0,309597155.0,110885540.0,151147457.0,196475151.0,1111051500
376625,20242,11110515,청운효자동,CS100009,호프-간이주점,367574819.0,11865.0,238687672.0,128887147.0,138737522.0,190379905.0,1111051500


In [15]:
# 모든 서비스 매출 합치기

index_cols = [
    "기준_년분기_코드",
    "행정동코드"
]

sum_cols = [
    "당월_매출_금액",
    "당월_매출_건수",
    "주중_매출_금액",
    "주말_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액"
]


sales_df = sales_df.groupby(index_cols,as_index=False)[sum_cols].sum()


print(f"갯수: {len(sales_df)}")
sales_df.head(2)



갯수: 2972


,기준_년분기_코드,행정동코드,당월_매출_금액,당월_매출_건수,주중_매출_금액,주말_매출_금액,남성_매출_금액,여성_매출_금액
0,20242,1111051500,2.727106e+10,1285256.0,2.039912e+10,6.871944e+09,9.044101e+09,1.592994e+10
1,20242,1111053000,1.039976e+11,4569507.0,8.323463e+10,2.076299e+10,4.034940e+10,4.093106e+10


In [17]:
# interpolate를 통해서 월단위 데이터 전환

# 분기 -> 월 매핑
quarter_month_map = {
    "1": "03",
    "2": "06",
    "3": "09",
    "4": "12"
}

value_cols = [
    "당월_매출_금액",
    "당월_매출_건수",
    "주중_매출_금액",
    "주말_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액"
]

code_cols = [
    "기준_년분기_코드",
    "행정동코드"
]


# YYYYQ -> YYYYMM 변환
sales_df["YYYYMM"] = (
    sales_df["기준_년분기_코드"]
    .astype(str)
    .str[:4]
    +
    sales_df["기준_년분기_코드"]
    .astype(str)
    .str[-1]
    .map(quarter_month_map)
)

result = []

# 행정동별 월 보간
for code, g in sales_df.groupby("행정동코드"):

    g = g.copy()

    # 날짜 변환
    g["DATE"] = pd.to_datetime(
        g["YYYYMM"],
        format="%Y%m"
    )

    g = g.sort_values("DATE")

    # index 설정
    g = g.set_index("DATE")


    # 월 단위 확장
    monthly = g.resample("MS").asfreq()

    # 수치 컬럼 보간
    monthly[value_cols] = (
        monthly[value_cols]
        .interpolate(method="linear")
    )

    # code 채우기
    monthly[code_cols] = (
        monthly[code_cols]
        .ffill()
    )

    # 행정동코드 유지
    monthly["행정동코드"] = code

    # YYYYMM 생성
    monthly["YYYYMM"] = (
        monthly.index.strftime("%Y%m")
    )

    result.append(monthly)


# 합치기
monthly_sales_df = (
    pd.concat(result)
    .reset_index(drop=True)
)


monthly_sales_df[value_cols] = (
    monthly_sales_df[value_cols]
    .round()
    .fillna(0)
    .astype(int)

)

monthly_sales_df[code_cols] = (
    monthly_sales_df[code_cols]
    .ffill()
)

monthly_sales_df = (
    monthly_sales_df
    .sort_values(
        ["YYYYMM", "행정동코드"]
    )
    .reset_index(drop=True)
)

monthly_sales_df[[
    "기준_년분기_코드",
    "YYYYMM",
    "행정동코드",
    "당월_매출_금액",
    "당월_매출_건수",
    "주중_매출_금액",
    "주말_매출_금액",
    "남성_매출_금액",
    "여성_매출_금액"]].to_csv(
    "서울시_행정동_매출_월단위_202404_202512.csv",
    index=False,
    encoding="utf-8-sig")


print(f"갯수: {len(monthly_sales_df)}")
monthly_sales_df.head(2)



갯수: 8066


,기준_년분기_코드,행정동코드,당월_매출_금액,당월_매출_건수,주중_매출_금액,주말_매출_금액,남성_매출_금액,여성_매출_금액,YYYYMM
0,20242.0,1111051500,27271062670,1285256,20399118315,6871944355,9044101078,15929938481,202406
1,20242.0,1111053000,103997623389,4569507,83234631676,20762991713,40349400561,40931063369,202406
